# Stock Predictions Delta Table Management

This notebook provides code snippets for managing the stock_predictions Delta table, including:
- Reading the Delta table
- Displaying all data
- Filtering by stock symbol
- Deleting table rows

## Prerequisites
- Spark ML pipeline environment with Delta Lake support
- stock_predictions Delta table created by the batch inference script

## 1. Import Required Libraries

Import Delta-Spark, PySpark, and other necessary libraries for working with Delta tables.

In [1]:
# Import Required Libraries
from pyspark.sql import SparkSession
from delta.tables import DeltaTable
from pyspark.sql.functions import col, count, desc, asc
import os
import warnings

# Suppress warnings for cleaner output
warnings.filterwarnings("ignore")

# Create Spark session with Delta Lake support
spark = SparkSession.builder \
    .appName("StockPredictionsManagement") \
    .config("spark.jars.packages", "io.delta:delta-spark_2.13:4.0.0") \
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension") \
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog") \
    .getOrCreate()

print("✅ Spark session created with Delta Lake support")
print(f"🔧 Spark version: {spark.version}")

# Set the path to your stock_predictions Delta table
DELTA_TABLE_PATH = "/home/mha2cob/Downloads/breaking_data/spark_ml_pipeline/delta_tables/stock_predictions"
print(f"📂 Delta table path: {DELTA_TABLE_PATH}")

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
25/11/26 01:02:55 WARN Utils: Your hostname, KOR-C-0159R, resolves to a loopback address: 127.0.1.1; using 192.168.29.79 instead (on interface wlp0s20f3)
25/11/26 01:02:55 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
:: loading settings :: url = jar:file:/home/mha2cob/anaconda3/envs/breaking_data/lib/python3.12/site-packages/pyspark/jars/ivy-2.5.3.jar!/org/apache/ivy/core/settings/ivysettings.xml
Ivy Default Cache set to: /home/mha2cob/.ivy2.5.2/cache
The jars for the packages stored in: /home/mha2cob/.ivy2.5.2/jars
io.delta#delta-spark_2.13 added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-8719a125-c443-492e-82eb-9c6c6845ba1d;1.0
	confs: [default]
	found io.delta#delta-spark_2.13;4.0.0 in central
	found io.delta#delta-storage;4.0.0 in central
	found org.antlr#antlr4-runtime;4.13.1 in central
:: resolution report :: resolve 357ms :: artifacts d

✅ Spark session created with Delta Lake support
🔧 Spark version: 4.0.1
📂 Delta table path: /home/mha2cob/Downloads/breaking_data/spark_ml_pipeline/delta_tables/stock_predictions


## 2. Read Delta Table

Load the stock_predictions Delta table and examine its structure.

In [2]:
# Method 1: Read using DeltaTable.forPath() (recommended for Delta-specific operations)
try:
    delta_table = DeltaTable.forPath(spark, DELTA_TABLE_PATH)
    print("✅ Delta table loaded successfully using DeltaTable.forPath()")
    
    # Convert to DataFrame for analysis
    df = delta_table.toDF()
    
except Exception as e:
    print(f"❌ Error loading Delta table: {e}")
    df = None

# Method 2: Read using Spark DataFrame API (alternative approach)
try:
    df_alt = spark.read.format("delta").load(DELTA_TABLE_PATH)
    print("✅ Delta table loaded successfully using DataFrame API")
except Exception as e:
    print(f"❌ Error loading with DataFrame API: {e}")

# Check if table exists and show basic info
if df is not None:
    print(f"\n📊 Table Schema:")
    df.printSchema()
    
    print(f"\n📈 Basic Table Statistics:")
    print(f"   • Total columns: {len(df.columns)}")
    print(f"   • Column names: {df.columns}")
else:
    print("⚠️  Could not load the Delta table")

✅ Delta table loaded successfully using DeltaTable.forPath()
✅ Delta table loaded successfully using DataFrame API

📊 Table Schema:
root
 |-- prediction_date: date (nullable = true)
 |-- prediction_timestamp: timestamp (nullable = true)
 |-- predicted_close: double (nullable = true)
 |-- model_confidence: double (nullable = true)
 |-- model_version: integer (nullable = true)
 |-- features_used: string (nullable = true)
 |-- created_at: timestamp (nullable = true)
 |-- days_ahead: integer (nullable = true)
 |-- stock_symbol: string (nullable = true)


📈 Basic Table Statistics:
   • Total columns: 9
   • Column names: ['prediction_date', 'prediction_timestamp', 'predicted_close', 'model_confidence', 'model_version', 'features_used', 'created_at', 'days_ahead', 'stock_symbol']


## 3. Display All Data

Show all records in the stock_predictions table and get summary statistics.

In [3]:
# Get total record count
total_count = df.count()
print(f"📊 Total predictions in table: {total_count}")

if total_count > 0:
    # Display all data (limit to first 20 rows for readability)
    print(f"\n📋 All Predictions (showing first 20 rows):")
    df.orderBy(col("prediction_date").desc(), col("stock_symbol")).show(20, truncate=False)
    
    # Summary by stock symbol
    print(f"\n🏢 Predictions by Stock Symbol:")
    stock_summary = df.groupBy("stock_symbol") \
                     .agg(count("*").alias("prediction_count")) \
                     .orderBy(desc("prediction_count"))
    stock_summary.show()
    
    # Summary by prediction date
    print(f"\n📅 Predictions by Date:")
    date_summary = df.groupBy("prediction_date") \
                    .agg(count("*").alias("prediction_count")) \
                    .orderBy(desc("prediction_date"))
    date_summary.show()
    
    # Model version summary
    print(f"\n🤖 Model Version Summary:")
    model_summary = df.groupBy("model_version") \
                     .agg(count("*").alias("prediction_count")) \
                     .orderBy("model_version")
    model_summary.show()
    
    # Quick statistics on predicted close prices
    print(f"\n💰 Predicted Close Price Statistics:")
    df.describe("predicted_close").show()
    
else:
    print("📭 No data found in the table")

25/11/26 01:03:29 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


📊 Total predictions in table: 100

📋 All Predictions (showing first 20 rows):


+---------------+--------------------+------------------+----------------+-------------+-------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+--------------------------+----------+------------+
|prediction_date|prediction_timestamp|predicted_close   |model_confidence|model_version|features_used                                                                                                                                                                                                                                                                                                                                                            |created_at                |days_ahead|stock_symb

## 3.5. Show Last Row for Each Stock

Display the most recent prediction for each stock symbol in the table.

In [14]:
# Get the last row (most recent prediction) for each stock symbol
print("🔍 Last Row for Each Stock Symbol:")
print("=" * 50)

if total_count > 0:
    # Method 1: Using window functions to get the latest row per stock
    from pyspark.sql.window import Window
    from pyspark.sql.functions import row_number, max as spark_max
    
    # Create window partitioned by stock_symbol, ordered by prediction_date and created_at descending
    window_spec = Window.partitionBy("stock_symbol").orderBy(
        col("prediction_date").desc(), 
        col("created_at").desc()
    )
    
    # Add row number and filter to get only the first row (most recent) for each stock
    latest_predictions = df.withColumn("row_num", row_number().over(window_spec)) \
                          .filter(col("row_num") == 1) \
                          .drop("row_num") \
                          .orderBy("stock_symbol")
    
    latest_count = latest_predictions.count()
    print(f"📊 Found latest predictions for {latest_count} unique stocks")
    
    # Display the results
    print(f"\n📋 Most Recent Prediction per Stock:")
    latest_predictions.select(
        "stock_symbol", 
        "prediction_date", 
        "predicted_close", 
        "model_confidence",
        "model_version",
        "created_at"
    ).show(30,truncate=False)
    
    # Alternative Method 2: Get latest by grouping and joining back
    # This method finds the max date for each stock, then joins to get full row
    print(f"\n🔄 Alternative method - Latest prediction dates by stock:")
    latest_dates = df.groupBy("stock_symbol") \
                    .agg(spark_max("prediction_date").alias("max_date"),
                         spark_max("created_at").alias("max_created")) \
                    .orderBy("stock_symbol")
    
    latest_dates.show(truncate=False)
    
    # Show summary statistics of the latest predictions
    print(f"\n💰 Latest Predictions - Price Statistics:")
    latest_predictions.describe("predicted_close").show()
    
    # Show distribution by prediction date
    print(f"\n📅 Distribution of Latest Prediction Dates:")
    latest_predictions.groupBy("prediction_date") \
                     .agg(count("*").alias("stock_count")) \
                     .orderBy(col("prediction_date").desc()) \
                     .show()
                     
else:
    print("📭 No data available to show latest predictions")

🔍 Last Row for Each Stock Symbol:
📊 Found latest predictions for 25 unique stocks

📋 Most Recent Prediction per Stock:
+------------+---------------+------------------+----------------+-------------+--------------------------+
|stock_symbol|prediction_date|predicted_close   |model_confidence|model_version|created_at                |
+------------+---------------+------------------+----------------+-------------+--------------------------+
|AAL         |2023-12-18     |14.60312220009298 |0.85            |1            |2025-11-24 23:14:49.997428|
|AAPL        |2023-12-18     |198.18346602832312|0.85            |1            |2025-11-24 23:14:49.997428|
|ABBV        |2023-12-18     |154.90437535466614|0.85            |1            |2025-11-24 23:14:49.997428|
|AMD         |2023-12-18     |137.96109985414992|0.85            |1            |2025-11-24 23:14:49.997428|
|AMGN        |2023-12-18     |276.48908368973474|0.85            |1            |2025-11-24 23:14:49.997428|
|BABA        |202

## 4. Filter Data by Stock Symbol

Filter the Delta table data for a specific stock symbol and display the filtered results.

In [6]:
# Define the stock symbol to filter for
STOCK_SYMBOL = "AAPL"  # Change this to any stock symbol you want to filter

print(f"🔍 Filtering predictions for stock: {STOCK_SYMBOL}")

# Method 1: Using .filter() method
filtered_df = df.filter(col("stock_symbol") == STOCK_SYMBOL)

# Method 2: Using .where() method (equivalent to filter)
# filtered_df = df.where(col("stock_symbol") == STOCK_SYMBOL)

# Method 3: Using SQL string expression
# filtered_df = df.filter(f"stock_symbol = '{STOCK_SYMBOL}'")

# Show filtered results
filtered_count = filtered_df.count()
print(f"📊 Found {filtered_count} predictions for {STOCK_SYMBOL}")

if filtered_count > 0:
    print(f"\n📋 {STOCK_SYMBOL} Predictions:")
    filtered_df.orderBy(col("prediction_date").desc()).show(truncate=False)
    
    # Show detailed statistics for this stock
    print(f"\n💰 {STOCK_SYMBOL} Predicted Close Price Statistics:")
    filtered_df.describe("predicted_close").show()
    
    # Show prediction history over time
    print(f"\n📈 {STOCK_SYMBOL} Prediction Timeline:")
    filtered_df.select("prediction_date", "predicted_close", "model_confidence", "created_at") \
               .orderBy(col("prediction_date").asc()) \
               .show(truncate=False)
else:
    print(f"❌ No predictions found for stock symbol: {STOCK_SYMBOL}")

# Show available stock symbols for reference
print(f"\n📑 Available Stock Symbols in the table:")
available_stocks = df.select("stock_symbol").distinct().orderBy("stock_symbol").collect()
stock_list = [row.stock_symbol for row in available_stocks]
print(f"   {stock_list}")

🔍 Filtering predictions for stock: AAPL
📊 Found 4 predictions for AAPL

📋 AAPL Predictions:
+---------------+--------------------+------------------+----------------+-------------+-------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+--------------------------+----------+------------+
|prediction_date|prediction_timestamp|predicted_close   |model_confidence|model_version|features_used                                                                                                                                                                                                                                                                                                                 

## 5. Delete Operations

Use Delta table's delete operation to remove rows from the stock_predictions table.

### 5a. Delete Specific Stock Predictions

Delete predictions for a specific stock symbol (safer than deleting all data).

In [ ]:
# CAUTION: This will delete data. Uncomment and run only if you're sure!
# STOCK_TO_DELETE = "AAPL"  # Change this to the stock you want to delete

# # Check current count before deletion
# current_count = df.count()
# stock_count_before = df.filter(col("stock_symbol") == STOCK_TO_DELETE).count()

# print(f"📊 Current total predictions: {current_count}")
# print(f"📊 Predictions for {STOCK_TO_DELETE}: {stock_count_before}")

# # Perform the deletion
# if stock_count_before > 0:
#     print(f"\n⚠️  DELETING {stock_count_before} predictions for {STOCK_TO_DELETE}...")
#     
#     # Delete specific stock predictions
#     delta_table.delete(condition = col("stock_symbol") == STOCK_TO_DELETE)
#     
#     # Verify deletion
#     new_total_count = delta_table.toDF().count()
#     remaining_stock_count = delta_table.toDF().filter(col("stock_symbol") == STOCK_TO_DELETE).count()
#     
#     print(f"✅ Deletion completed!")
#     print(f"📊 New total predictions: {new_total_count}")
#     print(f"📊 Remaining predictions for {STOCK_TO_DELETE}: {remaining_stock_count}")
# else:
#     print(f"❌ No predictions found for {STOCK_TO_DELETE} to delete")

print("💡 To delete specific stock predictions:")
print("   1. Uncomment the code above")
print("   2. Set STOCK_TO_DELETE to your desired stock symbol") 
print("   3. Run the cell")

### 5b. Delete All Table Rows

⚠️ **DANGER ZONE** ⚠️ Delete ALL predictions from the table.

In [ ]:
# ⚠️  EXTREME CAUTION: This will delete ALL data in the table!
CONFIRM_DELETE_ALL = True  # Set to True if you really want to delete everything

if CONFIRM_DELETE_ALL:
    print("🚨 DELETING ALL PREDICTIONS...")
    
    # Check current count
    current_count = delta_table.toDF().count()
    print(f"📊 Current total predictions: {current_count}")
    
    if current_count > 0:
        # Delete all rows (no condition means delete everything)
        delta_table.delete()
        
        # Verify deletion
        final_count = delta_table.toDF().count()
        print(f"✅ Deletion completed!")
        print(f"📊 Remaining predictions: {final_count}")
        
        if final_count == 0:
            print("🎯 All predictions successfully deleted")
        else:
            print(f"⚠️  Warning: {final_count} predictions still remain")
    else:
        print("📭 Table is already empty")
else:
    print("🛡️  DELETE ALL operation is DISABLED for safety")
    print("   To enable:")
    print("   1. Set CONFIRM_DELETE_ALL = True")
    print("   2. Run this cell")
    print("   ⚠️  WARNING: This will permanently delete ALL predictions!")

print("\n💡 Alternative: Delete by date range or other conditions")
print("   Example: delta_table.delete(col('prediction_date') < '2023-01-01')")
print("   Example: delta_table.delete(col('model_version') == '1')")

## 6. Table History and Maintenance

View Delta table history and perform maintenance operations.

In [ ]:
# View Delta table history
print("📚 Delta Table History (last 10 operations):")
history_df = delta_table.history(10)
history_df.select("version", "timestamp", "operation", "operationParameters", "readVersion").show(truncate=False)

# Show table details
print(f"\n📋 Delta Table Details:")
detail_df = delta_table.detail()
detail_df.show(truncate=False)

# Check file statistics
print(f"\n📁 Table File Statistics:")
try:
    # This shows statistics about the underlying files
    spark.sql(f"DESCRIBE DETAIL delta.`{DELTA_TABLE_PATH}`").show(truncate=False)
except Exception as e:
    print(f"Could not get detailed statistics: {e}")

# Vacuum operation (removes old files - use with caution)
print(f"\n🧹 Table Maintenance:")
print("   • To vacuum old files: delta_table.vacuum(168)  # Remove files older than 7 days")
print("   • To optimize layout: delta_table.optimize().executeCompaction()")
print("   • To collect statistics: spark.sql(f'ANALYZE TABLE delta.`{DELTA_TABLE_PATH}` COMPUTE STATISTICS')")

# Example of checking table properties
print(f"\n🔍 Table Properties:")
try:
    properties = spark.sql(f"SHOW TBLPROPERTIES delta.`{DELTA_TABLE_PATH}`")
    properties.show(truncate=False)
except Exception as e:
    print(f"Could not retrieve table properties: {e}")

## 7. Cleanup

Stop the Spark session when done.

In [ ]:
# Stop the Spark session
print("🛑 Stopping Spark session...")
spark.stop()
print("✅ Spark session stopped successfully")

print("\n📝 Summary of operations performed:")
print("   ✓ Loaded stock_predictions Delta table")
print("   ✓ Displayed table schema and statistics") 
print("   ✓ Showed all predictions data")
print("   ✓ Filtered data by stock symbol")
print("   ✓ Provided safe deletion examples")
print("   ✓ Showed table history and maintenance options")
print("\n🎯 Use this notebook to manage your stock predictions data safely!")